# Haus & Haus — Milvus Property Vector Search

End-to-end notebook for indexing and searching property listings in the Milvus collection **`property_listings`**.

**Embedding model:** `BAAI/bge-small-en-v1.5`  
**Vector dimension:** 384  
**Source CSV:** `data/processed/property_listings_raw.csv`

> Safety: this notebook does not drop or recreate an existing collection. If the collection already exists with an incompatible schema, stop and inspect it rather than deleting data.

## 0. Install dependencies

Run this once in the terminal from the project root:

```bash
cd ~/momina/projects/real-estate-project
uv add fastembed pymilvus python-dotenv pandas
```

In VS Code, select the project's `.venv` Python kernel for this notebook.

## 1. Imports and configuration

Keep credentials in `.env`, never directly in the notebook. The collection name is intentionally fixed to `property_listings`.

In [2]:
from pathlib import Path
import os
import math
import pandas as pd
from dotenv import load_dotenv
from fastembed import TextEmbedding
from pymilvus import MilvusClient, DataType

In [5]:
PROJECT_ROOT = Path.cwd().parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "property_listings_raw.csv"
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

COLLECTION_NAME = "property_listings"
MODEL_NAME = "BAAI/bge-small-en-v1.5"
VECTOR_DIM = 384

MILVUS_URI = os.getenv("MILVUS_URI")
MILVUS_TOKEN = os.getenv("MILVUS_TOKEN")  # optional for local Milvus; often required for managed/cloud instances

print("Project root:", PROJECT_ROOT.resolve())
print("CSV exists:", CSV_PATH.exists())
print("Collection:", COLLECTION_NAME)
print("Embedding model:", MODEL_NAME)
print("MILVUS_URI configured:", bool(MILVUS_URI))

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Run the notebook from the project root "
        "or update PROJECT_ROOT / CSV_PATH."
    )

if not MILVUS_URI:
    raise ValueError(
        "MILVUS_URI is missing. Add it to your project's .env file. "
        "Do not paste your token or credentials into notebook cells."
    )

Project root: /home/ekkel-ai/momina/projects/real-estate-project
CSV exists: True
Collection: property_listings
Embedding model: BAAI/bge-small-en-v1.5
MILVUS_URI configured: True


### Expected `.env` settings

Use the URI/token for the Milvus instance you have already configured. For a managed instance, this usually resembles:

```dotenv
MILVUS_URI=https://your-milvus-endpoint
MILVUS_TOKEN=your-token
```

For a local Milvus Lite database, `MILVUS_URI` can instead be a local database path such as `./milvus.db`; a token is generally not needed. Do not replace working connection values blindly.

## 2. Connect to Milvus

In [6]:
connection_args = {"uri": MILVUS_URI}
if MILVUS_TOKEN:
    connection_args["token"] = MILVUS_TOKEN

client = MilvusClient(**connection_args)

print("Connected to Milvus.")
print("Collection exists:", client.has_collection(collection_name=COLLECTION_NAME))
print("Collections visible to this connection:", client.list_collections())

Connected to Milvus.
Collection exists: False
Collections visible to this connection: []


## 3. Load and inspect the raw CSV

The raw CSV remains unchanged. We create a working DataFrame for the vector index.

In [7]:
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Shape:", df.shape)
display(df.head(3))

expected_columns = [
    "id", "title", "price", "beds", "bathrooms", "community",
    "property_type", "listing_type", "status", "description",
    "long_description", "link"
]
print("Missing expected columns:", [c for c in expected_columns if c not in df.columns])

Shape: (6495, 57)


,id,crm_id,strapi_id,slug,title,listing_type,department,search_type,status,price,...,agent_phone,agent_whatsapp,agent_designation,agent_languages,agent_photo_url,agent_slug,video_tour,virtual_tour,enriched_at,property_type
0,9971,DBI215069,9971,2-bedroom-apartment-for-sale-in-mosela-watersi...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2850000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T17:03:40Z,apartment
1,12292,DUB203155,12292,3-bedroom-villa-for-sale-in-district-8-jumeira...,3 bedroom villa for sale,purchase,residential,sales,For Sale,7195000.0,...,+971 50 598 0269,+971 50 598 0269,Associate Partner,"[""English""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,cathal-o-neill,[],[],2026-09-17T18:04:47Z,villa
2,10051,DBI210323,10051,2-bedroom-apartment-for-sale-in-al-ghozlan-1-t...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2250000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T18:04:47Z,apartment


Missing expected columns: []


In [8]:
# Basic data-quality checks
print("Rows:", len(df))
if "id" in df.columns:
    print("Missing property IDs:", df["id"].isna().sum())
    print("Duplicate property IDs:", df["id"].astype("string").duplicated().sum())

for col in ["price", "beds", "bathrooms"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

display(df[["price", "beds", "bathrooms"]].describe().T)

Rows: 6495
Missing property IDs: 0
Duplicate property IDs: 0


,count,mean,std,min,25%,50%,75%,max
price,6495.0,1.739161e+06,6.242300e+06,0.0,115000.0,200000.0,875000.0,249000000.0
beds,6495.0,2.236182e+00,1.458580e+00,0.0,1.0,2.0,3.0,9.0
bathrooms,6495.0,2.812779e+00,1.603351e+00,0.0,2.0,3.0,4.0,16.0


## 4. Prepare property text and metadata

Semantic embeddings are for meaning and relevance. Exact constraints such as price, bedroom count, listing type and status should be applied as metadata filters—not left to the embedding model.

This creates a compact search document from fields that are useful to a property seeker. We deliberately avoid embedding every CSV column.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    # Basic HTML tag removal and whitespace normalization.
    import re
    value = str(value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"&nbsp;|&#160;", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

def text_value(row, column):
    return clean_text(row[column]) if column in row.index else ""

def make_search_text(row):
    parts = [
        f"Title: {text_value(row, 'title')}",
        f"Property type: {text_value(row, 'property_type')}",
        f"Community: {text_value(row, 'community')}",
        f"Building: {text_value(row, 'building')}",
        f"Bedrooms: {text_value(row, 'beds')}",
        f"Bathrooms: {text_value(row, 'bathrooms')}",
        f"Description: {text_value(row, 'description')}",
        f"Additional details: {text_value(row, 'accommodation_summary')}",
        f"Long description: {text_value(row, 'long_description')}",
    ]
    return " ".join(part for part in parts if part.split(": ", 1)[-1].strip())

# Stable property ID is required for deduplication and repeatable indexing.
df["property_id"] = df["id"].astype("string").str.strip()
df = df[df["property_id"].notna() & (df["property_id"] != "")].copy()
df = df.drop_duplicates(subset=["property_id"], keep="last").copy()

df["search_text"] = df.apply(make_search_text, axis=1)
df = df[df["search_text"].str.strip().ne("")].copy()

print("Rows eligible for indexing:", len(df))
display(df[["property_id", "title", "community", "price", "beds", "search_text"]].head(3))

## 5. Initialize the embedding model and test one vector

The BGE query instruction is used for **search queries**. For property documents, embed the prepared property text without that query prefix.

In [ ]:
embedding_model = TextEmbedding(model_name=MODEL_NAME)

sample_text = df.iloc[0]["search_text"]
sample_vector = list(embedding_model.embed([sample_text]))[0]

print("Vector dimensions:", len(sample_vector))
print("First 5 vector values:", sample_vector[:5])

assert len(sample_vector) == VECTOR_DIM, (
    f"Expected {VECTOR_DIM} dimensions, got {len(sample_vector)}"
)

## 6. Define the Milvus schema

Schema fields:
- `property_id`: stable primary key from the source `id`
- `embedding`: 384-dimensional vector
- typed scalar metadata used for filters
- short text fields returned in search results

If `property_listings` already exists, the notebook inspects its schema and does **not** delete it. If it is incompatible, stop here and decide on a migration or a new collection strategy.

In [ ]:
def build_collection_schema():
    schema = client.create_schema(auto_id=False, enable_dynamic_field=False)

    schema.add_field(
        field_name="property_id",
        datatype=DataType.VARCHAR,
        is_primary=True,
        max_length=128,
    )
    schema.add_field(
        field_name="embedding",
        datatype=DataType.FLOAT_VECTOR,
        dim=VECTOR_DIM,
    )
    schema.add_field(field_name="price", datatype=DataType.DOUBLE)
    schema.add_field(field_name="beds", datatype=DataType.INT64)
    schema.add_field(field_name="bathrooms", datatype=DataType.DOUBLE)
    schema.add_field(field_name="community", datatype=DataType.VARCHAR, max_length=512)
    schema.add_field(field_name="property_type", datatype=DataType.VARCHAR, max_length=256)
    schema.add_field(field_name="listing_type", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="status", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=1024)
    schema.add_field(field_name="link", datatype=DataType.VARCHAR, max_length=2048)
    schema.add_field(field_name="search_text", datatype=DataType.VARCHAR, max_length=8192)
    return schema

if client.has_collection(collection_name=COLLECTION_NAME):
    existing_description = client.describe_collection(collection_name=COLLECTION_NAME)
    print("Collection already exists; not recreating it.")
    display(existing_description)
    print("\nInspect the fields above. The expected schema is shown in the previous markdown cell.")
else:
    schema = build_collection_schema()
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="embedding",
        index_type="AUTOINDEX",
        metric_type="COSINE",
    )
    client.create_collection(
        collection_name=COLLECTION_NAME,
        schema=schema,
        index_params=index_params,
    )
    print(f"Created collection: {COLLECTION_NAME}")

print("Collection ready check:", client.has_collection(collection_name=COLLECTION_NAME))

## 7. Validate the collection schema before writing

Run this cell even if the collection already existed. It checks that required fields are present and the vector dimension matches. It intentionally fails rather than silently deleting/replacing an incompatible collection.

In [ ]:
description = client.describe_collection(collection_name=COLLECTION_NAME)
fields_info = description.get("fields", [])
field_map = {field.get("name"): field for field in fields_info}

print("Field names:", list(field_map.keys()))

required_fields = {
    "property_id", "embedding", "price", "beds", "bathrooms",
    "community", "property_type", "listing_type", "status",
    "title", "link", "search_text"
}
missing_fields = required_fields - set(field_map)

if missing_fields:
    raise RuntimeError(
        f"Existing collection '{COLLECTION_NAME}' is missing required fields: "
        f"{sorted(missing_fields)}. Do not drop it automatically. Inspect the schema "
        "and choose a migration or a separately named test collection."
    )

vector_field = field_map["embedding"]
vector_params = vector_field.get("params", {})
if vector_params and int(vector_params.get("dim", VECTOR_DIM)) != VECTOR_DIM:
    raise RuntimeError(
        f"Collection vector dimension is incompatible: {vector_params.get('dim')} "
        f"(expected {VECTOR_DIM}). Do not drop it automatically."
    )

print("Required fields appear present. Confirm the displayed schema matches this notebook before insertion.")

## 8. Convert rows into Milvus records

Missing numeric values are represented as `0` for price/beds/bathrooms in this first version. For business-critical filtering, treat these values as unknown and exclude them from matches requiring that field. Strings are trimmed to fit schema limits.

In [ ]:
def safe_number(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        value = float(value)
        if not math.isfinite(value):
            return default
        return value
    except (TypeError, ValueError):
        return default

def safe_int(value, default=0):
    number = safe_number(value, default)
    return int(number) if number >= 0 else default

def safe_string(value, max_length, default=""):
    value = clean_text(value)
    return (value or default)[:max_length]

def row_to_record(row, vector):
    price = safe_number(row.get("price"), 0.0)
    beds = safe_int(row.get("beds"), 0)
    bathrooms = safe_number(row.get("bathrooms"), 0.0)

    return {
        "property_id": str(row["property_id"])[:128],
        "embedding": vector.tolist() if hasattr(vector, "tolist") else list(vector),
        "price": price,
        "beds": beds,
        "bathrooms": bathrooms,
        "community": safe_string(row.get("community"), 512),
        "property_type": safe_string(row.get("property_type"), 256),
        "listing_type": safe_string(row.get("listing_type"), 128),
        "status": safe_string(row.get("status"), 128),
        "title": safe_string(row.get("title"), 1024),
        "link": safe_string(row.get("link"), 2048),
        "search_text": safe_string(row.get("search_text"), 8192),
    }

## 9. Embed the properties in batches

This can take a few minutes on the first run. Batches keep memory usage manageable. This cell prepares records in memory; the next cell writes them to Milvus.

In [ ]:
BATCH_SIZE = 128
records = []
texts = df["search_text"].tolist()
rows = [row for _, row in df.iterrows()]

for start in range(0, len(texts), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(texts))
    vectors = list(embedding_model.embed(texts[start:end]))
    for row, vector in zip(rows[start:end], vectors):
        records.append(row_to_record(row, vector))
    if start == 0 or end == len(texts) or end % (BATCH_SIZE * 5) == 0:
        print(f"Embedded {end:,} / {len(texts):,}")

print("Prepared records:", len(records))
print("Vector dimension:", len(records[0]["embedding"]) if records else "no records")

## 10. Insert or safely upsert records

Milvus `upsert` inserts new primary keys and updates records with matching primary keys. It is convenient for repeat runs when source IDs are stable. If the collection schema validation above failed, do not proceed.

In [ ]:
WRITE_BATCH_SIZE = 128
written = 0

for start in range(0, len(records), WRITE_BATCH_SIZE):
    batch = records[start:start + WRITE_BATCH_SIZE]
    result = client.upsert(
        collection_name=COLLECTION_NAME,
        data=batch,
    )
    written += len(batch)
    print(f"Upserted {written:,} / {len(records):,}", "| result:", result)

client.flush(collection_name=COLLECTION_NAME)
print("Write complete.")

## 11. Load the collection and verify the index

Loading makes the collection available for search in server-based Milvus deployments. If the server reports that the collection is already loaded, that is usually fine.

In [ ]:
client.load_collection(collection_name=COLLECTION_NAME)
print("Collection loaded.")

stats = client.get_collection_stats(collection_name=COLLECTION_NAME)
display(stats)

## 12. Semantic search

Use the BGE retrieval instruction on the **query**. Results are ranked by cosine similarity. Similarity scores are useful for ranking but should not be interpreted as probabilities.

In [ ]:
def embed_query(query):
    query_for_embedding = (
        "Represent this sentence for searching relevant passages: " + query.strip()
    )
    vector = list(embedding_model.embed([query_for_embedding]))[0]
    return vector.tolist() if hasattr(vector, "tolist") else list(vector)

def search_properties(query, limit=10, filter_expr=None):
    query_vector = embed_query(query)
    results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        anns_field="embedding",
        limit=limit,
        filter=filter_expr,
        output_fields=[
            "property_id", "title", "price", "beds", "bathrooms",
            "community", "property_type", "listing_type", "status", "link"
        ],
        search_params={"metric_type": "COSINE", "params": {}},
    )

    hits = results[0] if results else []
    output = []
    for hit in hits:
        entity = hit.get("entity", {})
        output.append({
            "property_id": entity.get("property_id"),
            "title": entity.get("title"),
            "price": entity.get("price"),
            "beds": entity.get("beds"),
            "bathrooms": entity.get("bathrooms"),
            "community": entity.get("community"),
            "property_type": entity.get("property_type"),
            "listing_type": entity.get("listing_type"),
            "status": entity.get("status"),
            "link": entity.get("link"),
            "similarity_score": hit.get("distance"),
        })
    return pd.DataFrame(output)

query = "A spacious family home close to parks and family-friendly amenities"
results_df = search_properties(query, limit=10)
display(results_df)

## 13. Hard-filtered search examples

Milvus filter expressions use the exact field names in the collection schema. Check the dataset's actual `listing_type` and `status` values before relying on them in production.

In [ ]:
# Example: two-bedroom properties under AED 2,000,000.
# price > 0 excludes records where price was missing/zero in the source.
filter_expr = 'price > 0 and price <= 2000000 and beds == 2'

filtered_results = search_properties(
    "A modern apartment suitable for a small family",
    limit=10,
    filter_expr=filter_expr,
)
display(filtered_results)

In [ ]:
# Example: filter by community. Escape double quotes in the user-supplied value.
def escape_milvus_string(value):
    return str(value).replace("\\", "\\\\").replace('"', '\\"')

community_name = "Dubai Marina"
filter_expr = (
    f'price > 0 and community like "%{escape_milvus_string(community_name)}%"'
)

community_results = search_properties(
    "Apartment with a sea view and convenient local amenities",
    limit=10,
    filter_expr=filter_expr,
)
display(community_results)

## 14. Inspect distinct metadata values

Use this before deciding which status or listing-type values should represent active sale/rent inventory. Do not assume `status` semantics without inspecting the source.

In [ ]:
for col in ["listing_type", "status", "property_type"]:
    if col in df.columns:
        print(f"\n{col} — top values")
        display(df[col].astype("string").fillna("<missing>").value_counts().head(20).rename_axis(col).reset_index(name="count"))

## 15. Retrieve a property by its ID

This uses the primary key for an exact lookup, not semantic similarity.

In [ ]:
def get_property(property_id):
    rows = client.get(
        collection_name=COLLECTION_NAME,
        ids=[str(property_id)],
        output_fields=[
            "property_id", "title", "price", "beds", "bathrooms",
            "community", "property_type", "listing_type", "status", "link", "search_text"
        ],
    )
    return pd.DataFrame(rows)

# Example:
# display(get_property("PUT_A_REAL_PROPERTY_ID_HERE"))

## 16. Operational notes and next steps

- **Do not change the embedding model without re-embedding** all indexed property text.
- Keep hard constraints (price, beds, listing type/status) in metadata filters; use vectors for semantic relevance.
- Confirm `listing_type` and `status` meanings from actual data before production filtering.
- Missing price/bedroom values are currently stored as `0`; filters should exclude unknowns where necessary.
- This first version stores a compact subset of metadata. Add other fields only when the application needs them.
- For production, add a retrieval evaluation set and compare top-k results against expected relevant properties.
- Avoid dropping the existing collection casually. If its schema differs, inspect it and plan a migration.